# 🔊 SONYC Urban Sound Tagging - Data Ingest
### Powered by Elasticsearch Serverless + Mistral (Voxtral)

[SONYC-UST](https://zenodo.org/records/2590742) is a multilabel dataset from NYC's **urban acoustic sensor network**: thousands of 10-second street recordings, each tagged by citizen scientists for 23 fine sound classes (jackhammers, car horns, sirens, music, dogs, human voices...) grouped into 8 coarse categories.

**What this notebook does:**
1. Downloads `annotations.csv` + the class taxonomy straight from Zenodo (no key, ~2.6 MB)
2. Aggregates the per-annotator tags into one clean record per recording
3. Indexes everything into `nyc_sonyc_sound` so you can explore what the city *sounds* like
4. **(Optional)** transcribes the human-voice clips with **Mistral Voxtral** (speech-to-text)

---
> **Data source:** Cartwright et al., *SONYC Urban Sound Tagging (SONYC-UST)*, Zenodo record `2590742`, CC BY 4.0. Audio (`audio.tar.gz`) is 1.9 GB and optional - the annotations alone are plenty to build on.

## 0. Install dependencies

In [ ]:
!pip install elasticsearch requests pyyaml --quiet

In [ ]:
import csv, io, re, requests, yaml
from collections import defaultdict
from elasticsearch import Elasticsearch, helpers

print('✅ Imports ready')

## 1. Connect to Elastic Serverless
> **Elastic Cloud Console → Your Project → Connection Details**

In [ ]:
# 🔑 Fill these in
ELASTIC_ENDPOINT = 'https://your-project.es.region.aws.elastic.cloud'
ELASTIC_API_KEY  = 'your-elastic-api-key'

es = Elasticsearch(ELASTIC_ENDPOINT, api_key=ELASTIC_API_KEY)
print(f'✅ Connected to Elasticsearch {es.info()["version"]["number"]}')

## 2. Download the annotations and taxonomy from Zenodo

In [ ]:
REC = 'https://zenodo.org/records/2590742/files'
ann = requests.get(f'{REC}/annotations.csv', timeout=120); ann.raise_for_status()
rows = list(csv.DictReader(io.StringIO(ann.text)))

tax = yaml.safe_load(requests.get(f'{REC}/dcase-ust-taxonomy.yaml', timeout=60).text)
COARSE_NAMES = {   # 8 coarse categories
    '1': 'engine', '2': 'machinery-impact', '3': 'non-machinery-impact', '4': 'powered-saw',
    '5': 'alert-signal', '6': 'music', '7': 'human-voice', '8': 'dog'}

print(f'✅ {len(rows)} annotation rows, {len(rows[0])} columns')
# Fine-class presence columns look like '5-3_siren_presence'. The file also has
# 8 coarse-level columns like '5_alert-signal_presence' (no dash) — skip those.
FINE_RE = re.compile(r'^\d+-')
presence_cols = [c for c in rows[0] if c.endswith('_presence') and FINE_RE.match(c)]
print(f'   {len(presence_cols)} fine sound classes')

## 3. Aggregate per recording
Each clip is tagged by up to three annotators. We mark a class **present** if any annotator heard it, and roll the fine classes up to their 8 coarse categories.

In [ ]:
def fine_label(col):            # '5-3_siren_presence' -> 'siren'
    return col[:-len('_presence')].split('_', 1)[1]
def coarse_of(col):             # '5-3_siren_presence' -> 'alert-signal'
    return COARSE_NAMES.get(col.split('-', 1)[0], 'unknown')

agg = {}
annotators = defaultdict(set)
for r in rows:
    fn = r['audio_filename']
    rec = agg.setdefault(fn, {'audio_filename': fn, 'sensor_id': r.get('sensor_id'),
                              'split': r.get('split'), 'fine': set(), 'coarse': set()})
    annotators[fn].add(r.get('annotator_id'))
    for c in presence_cols:
        try:
            if float(r[c]) >= 1:
                rec['fine'].add(fine_label(c)); rec['coarse'].add(coarse_of(c))
        except (ValueError, TypeError):
            pass

docs = []
for fn, rec in agg.items():
    docs.append({'audio_filename': fn, 'sensor_id': rec['sensor_id'], 'split': rec['split'],
                 'fine_classes': sorted(rec['fine']), 'coarse_classes': sorted(rec['coarse']),
                 'num_sound_types': len(rec['fine']), 'num_annotators': len(annotators[fn])})
print(f'✅ Aggregated into {len(docs)} unique recordings')
print('   example:', {k: docs[0][k] for k in ('audio_filename','coarse_classes')})

## 4. Create index and ingest

In [ ]:
INDEX = 'nyc_sonyc_sound'
if es.indices.exists(index=INDEX):
    es.indices.delete(index=INDEX)
es.indices.create(index=INDEX, body={'mappings': {'properties': {
    'audio_filename':  {'type': 'keyword'},
    'sensor_id':       {'type': 'keyword'},
    'split':           {'type': 'keyword'},
    'fine_classes':    {'type': 'keyword'},
    'coarse_classes':  {'type': 'keyword'},
    'num_sound_types': {'type': 'integer'},
    'num_annotators':  {'type': 'integer'},
}}})

helpers.bulk(es, [{'_index': INDEX, '_source': d} for d in docs], raise_on_error=False)
es.indices.refresh(index=INDEX)
print(f'✅ Indexed {len(docs)} recordings into {INDEX}')

## 5. Verify - what does NYC sound like?

In [ ]:
resp = es.search(index=INDEX, size=0, body={'aggs': {
    'coarse': {'terms': {'field': 'coarse_classes', 'size': 10}},
    'fine':   {'terms': {'field': 'fine_classes', 'size': 10}},
}})
print(f"Total recordings: {resp['hits']['total']['value']}\n")
print('Most common coarse categories:')
for b in resp['aggregations']['coarse']['buckets']:
    print(f"  {b['key']:22s} {b['doc_count']}")
print('\nMost common fine classes:')
for b in resp['aggregations']['fine']['buckets']:
    print(f"  {b['key']:34s} {b['doc_count']}")

## 6. Going further with Mistral

The recordings tagged `human-voice` are street speech — a natural fit for **Mistral Voxtral** speech-to-text: transcribe the clips and index the transcript alongside the sound tags. The sound-class labels also pair well with a Mistral chat model for summarising the city's soundscape. See [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md).

---
## ✅ Index loaded
`nyc_sonyc_sound` captures the city's soundscape. Explore it with queries like "which sensors hear the most jackhammers?" or "find clips with both sirens and music", and layer in Voxtral transcription for the human-voice clips (see [using_mistral_in_elasticsearch.md](using_mistral_in_elasticsearch.md)).